# Module 06: RAG & Practical AI Systems

**Course:** Introduction to Modern AI  
**Estimated Time:** 90 minutes  
**Prerequisites:** Module 01-05

---

## 1. Why Retrieval-Augmented Generation (RAG)?

LLMs have fixed training cutoffs and can hallucinate on private datasets. RAG acts as an **open-book exam**: it searches an external vector database for relevant facts first, then injects them into the model's prompt.

In [ ]:
# Install dependencies for vector retrieval and generation
!pip install -q sentence-transformers transformers torch numpy

## 2. Building an End-to-End RAG Pipeline

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Custom Knowledge Base
kb_docs = [
    "NovaFlow refund policy: Full refunds are granted within 30 days of purchase if under 50 API credits.",
    "NovaFlow enterprise pricing: Enterprise tiers start at $499 per month with a 99.9% uptime SLA.",
    "NovaFlow data security: All customer datasets are encrypted at rest using AES-256 and via TLS 1.3 in transit.",
    "NovaFlow API limits: Free accounts are rate-limited to 60 requests per minute."
]

# 2. Load Models
print("Loading embedding and text generation models...")
retriever = SentenceTransformer('all-MiniLM-L6-v2')
gen_tokenizer = AutoTokenizer.from_pretrained('gpt2')
gen_model = AutoModelForCausalLM.from_pretrained('gpt2')
gen_model.eval()

# Index the knowledge base
doc_vectors = retriever.encode(kb_docs, convert_to_numpy=True)

# 3. Query & Generation Pipeline
def rag_query(user_question):
    # Semantic Search
    q_vec = retriever.encode(user_question, convert_to_numpy=True)
    scores = np.dot(doc_vectors, q_vec) / (np.linalg.norm(doc_vectors, axis=1) * np.linalg.norm(q_vec))
    best_idx = np.argmax(scores)
    context = kb_docs[best_idx]
    
    # Prompt Augmentation
    prompt = f"Context: {context}\nQuestion: {user_question}\nAnswer:"
    
    # Generation
    input_ids = gen_tokenizer(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        out = gen_model.generate(
            input_ids,
            max_new_tokens=25,
            pad_token_id=gen_tokenizer.eos_token_id
        )
    
    ans = gen_tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"Question: {user_question}")
    print(f"Retrieved Fact (Score: {scores[best_idx]:.4f}): {context}")
    print(f"Answer: {ans[len(prompt):].strip()}\n")

# Test Pipeline
rag_query("What is the encryption standard used for stored data?")
rag_query("How much does the enterprise plan cost per month?")